# 03 — Run TEMPTED once on the full cohort

**Input:** filtered counts and metadata from notebook 02  
**Does:** applies TEMPTED's half-minimum pseudocount + CLR and fits one five-component TEMPTED model using all eligible subjects  
**Output:** fitted subject component scores, genus loadings, and the saved TEMPTED model

There is no train/test split in this notebook.

In [ ]:
suppressPackageStartupMessages(library(tempted))

RANK <- 5
SMOOTH <- 1e-4
MAXITER <- 100
EPSILON <- 1e-4

root <- if (dir.exists("data")) "." else ".."
input_folder <- tail(sort(list.dirs(file.path(root, "data", "preprocessing"), recursive = FALSE)), 1)
output <- file.path(root, "data", "tempted", format(Sys.time(), "%Y%m%d_%H%M%S"))
dir.create(output, recursive = TRUE)

counts <- read.csv(file.path(input_folder, "counts_filtered.csv"), check.names = FALSE)
metadata <- read.csv(file.path(input_folder, "metadata.csv"), stringsAsFactors = FALSE)

rownames(counts) <- as.character(counts$sample_id)
counts$sample_id <- NULL
metadata$sample_id <- as.character(metadata$sample_id)
metadata$subject_id <- as.character(metadata$subject_id)
counts <- counts[metadata$sample_id, , drop = FALSE]

cat("Input:", input_folder, "\n")
cat("Output:", output, "\n")

## TEMPTED CLR

For each sample, the pseudocount is one-half of that sample's smallest positive retained count. The code then takes logs and subtracts the row mean.

In [ ]:
clr <- as.matrix(counts)

for (i in seq_len(nrow(clr))) {
  positive <- clr[i, clr[i, ] > 0]
  pseudocount <- min(positive) / 2
  logged <- log(clr[i, ] + pseudocount)
  clr[i, ] <- logged - mean(logged)
}

rownames(clr) <- rownames(counts)
colnames(clr) <- colnames(counts)

## Fit one full-cohort model

In [ ]:
tempted_data <- format_tempted(
  clr[metadata$sample_id, , drop = FALSE],
  metadata$age,
  metadata$subject_id,
  threshold = 1,
  transform = "none"
)

center <- svd_centralize(tempted_data, r = 1)
model <- tempted(
  center$datlist,
  r = RANK,
  smooth = SMOOTH,
  maxiter = MAXITER,
  epsilon = EPSILON
)

In [ ]:
A <- as.data.frame(model$A_hat)
names(A) <- paste0("component_", seq_len(ncol(A)))
A$subject_id <- rownames(A)

subject_country <- unique(metadata[c("subject_id", "country")])
subject_scores <- merge(A, subject_country, by = "subject_id")

B <- as.matrix(model$B_hat)
if (nrow(B) < ncol(B)) B <- t(B)

feature_id <- rownames(B)
if (is.null(feature_id)) feature_id <- colnames(counts)[seq_len(nrow(B))]

feature_loadings <- data.frame(feature_id = feature_id, B, check.names = FALSE)
names(feature_loadings)[-1] <- paste0("component_", seq_len(ncol(B)))

stopifnot(sum(grepl("^component_", names(subject_scores))) == RANK)
stopifnot(sum(grepl("^component_", names(feature_loadings))) == RANK)

write.csv(subject_scores, file.path(output, "subject_scores.csv"), row.names = FALSE)
write.csv(feature_loadings, file.path(output, "feature_loadings.csv"), row.names = FALSE)
saveRDS(model, file.path(output, "model.rds"))

settings <- data.frame(rank = RANK, smooth = SMOOTH, maxiter = MAXITER, epsilon = EPSILON)
write.csv(settings, file.path(output, "settings.csv"), row.names = FALSE)

cat("Saved:", output, "\n")
print(head(subject_scores))